# CIS 6211 – Foundations of Data Science
## Lab 2: Data Collection & Integration

**Student Name:** Rana Sultan Alhinidy  
**Course:** CIS 6211 | King Khalid University


## Objectives
- Load CSV and JSON/API data
- Inspect schemas
- Integrate datasets
- Document challenges

In [ ]:
import pandas as pd
import numpy as np
import json
import requests

pd.set_option("display.max_columns", None)

### 📌 Cell Explanation
This cell imports the required libraries:
- **pandas** is used for loading and manipulating structured data (DataFrames).
- **numpy** provides numerical operations.
- **json** is used to read and parse JSON files.
- **requests** allows making HTTP requests to fetch data from web APIs.
- `pd.set_option('display.max_columns', None)` ensures all columns are visible when printing a DataFrame.

## Step 1 – Load CSV Dataset

In [ ]:
airbnb_df = pd.read_csv("AB_NYC_2019.csv")
airbnb_df.head()

### 📌 Cell Explanation
The **Airbnb New York City 2019** dataset is loaded from a CSV file using `pd.read_csv()`. This dataset contains over 48,000 listings with 16 columns including host information, neighbourhood, room type, price, and availability.

The `head()` function displays the first 5 rows to give a quick overview of the data structure before any processing.

In [ ]:
airbnb_df.info()
airbnb_df.describe()

### 📌 Cell Explanation
- **`info()`** shows the data type of each column, the number of non-null values, and memory usage. This helps identify columns with missing values and incorrect data types.
- **`describe()`** provides descriptive statistics for all numeric columns (mean, standard deviation, min, max, and percentiles). This gives a first impression of the data distribution and potential outliers (e.g., price max = 10,000).

## Step 2 – Load JSON / API Data

In [ ]:
with open("weather_sample.json", "r") as f:
    weather_json = json.load(f)

weather_df = pd.json_normalize(weather_json)
weather_df.head()

### 📌 Cell Explanation
Weather data for New York City is loaded from a JSON file (originally from the **OpenWeatherMap API**).

- `json.load()` reads the file and converts it into a Python dictionary.
- `pd.json_normalize()` flattens the nested JSON structure into a flat table suitable for analysis. For example, the nested field `main.temp` becomes a column named `main.temp` in the DataFrame.

JSON data from APIs is typically nested, so normalization is a necessary step before merging with tabular data.

## Step 3 – Schema Inspection

In [ ]:
airbnb_df.columns
weather_df.columns

### 📌 Cell Explanation
The column names of both datasets are displayed before merging. This step is essential to:
1. Identify if there is a **common column** that can be used as a join key.
2. Detect **naming inconsistencies** between the two tables.

In this case, both datasets relate to New York City, but the city name is stored under different column names (`city` vs `name`), which requires a custom join specification.

## Step 4 – Data Integration

In [ ]:
airbnb_df["city"] = "New York"

merged_df = pd.merge(
    airbnb_df,
    weather_df,
    left_on="city",
    right_on="name",
    how="left"
)

merged_df.head()

### 📌 Cell Explanation
The two datasets are merged using `pd.merge()`:

1. A new column `city` with the value `'New York'` is added to the Airbnb data to create a join key.
2. The merge uses `left_on='city'` and `right_on='name'` because the city name is stored under different column names in each dataset.
3. `how='left'` means **all Airbnb rows are kept** even if there is no match in the weather data.

**Result:** Each Airbnb listing now also contains weather information (temperature, humidity, wind speed) for New York.

## Step 5 – Resolve Integration Issues

In [ ]:
merged_df.isnull().sum()

### 📌 Cell Explanation
Missing values are checked after the merge. `isnull()` produces a True/False table where True indicates a missing value, and `sum()` counts the missing values per column.

This check is essential because merge operations can introduce new missing values when there is no match between the two tables. In this case, since all listings are in New York and the weather data is also for New York, no new missing values are expected from the merge itself.

## Reflection Questions

### 1. What join key(s) did you use and why?
We used **`city`** as the join key. A new column called `city` with the value `'New York'` was created in the Airbnb dataset and matched with the `name` column in the weather dataset. This was the only common attribute between the two datasets since both are related to New York City.

### 2. What integration issues did you encounter?
The two datasets had different structures — the Airbnb data is a CSV with thousands of rows, while the weather data is a single JSON record. The city name was also stored under different column names (`city` vs `name`), requiring custom `left_on` and `right_on` parameters. Additionally, the weather data represents a single point in time, so all Airbnb listings receive the same weather values.

### 3. How reliable is the merged dataset?
The merged dataset has limitations. The weather data is a single snapshot and does not reflect the actual weather conditions when guests stayed. For a more reliable dataset, historical weather data matched by date with each booking would be needed.